In [ ]:
import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np
from Project_code_clean_file import *



Importing The Images

In [ ]:
import os

image_names = [f for f in os.listdir('archive') if f.endswith('.png')]
image_names.sort()  # sort alphabetically for consistency
print(image_names)  # verify the list looks right

Solving For Optimal Alpha Aross All 24 Images Using Supervised and Self-Supervised Methods

In [ ]:
from skimage.metrics import structural_similarity as ssim

def run_experiment(noisy_img, clean_img, image_name):
    plot_alphas = np.linspace(0.001, 0.2, 350)
    supervised_losses = []
    sure_losses = []
    r2r_losses = []
    n2n_losses = []
    
    for a in plot_alphas:
        u = function_TV_denoising_CP(noisy_img, a, 1000)
        supervised_losses.append(MSE(u, clean_img))
        sure_losses.append(SURE_loss(noisy_img, a))
        r2r_losses.append(R2R_loss(noisy_img, a))
        n2n_losses.append(neighbour2neighbour_loss(noisy_img, a))
    
    # normalise
    supervised_norm = np.array(supervised_losses) / np.max(supervised_losses)
    sure_norm = np.array(sure_losses) / np.max(sure_losses)
    r2r_norm = np.array(r2r_losses) / np.max(r2r_losses)
    n2n_norm = np.array(n2n_losses) / np.max(n2n_losses)
    
    # find minima
    sup_min_alpha = plot_alphas[np.argmin(supervised_norm)]
    sure_min_alpha = plot_alphas[np.argmin(sure_norm)]
    r2r_min_alpha = plot_alphas[np.argmin(r2r_norm)]
    n2n_min_alpha = plot_alphas[np.argmin(n2n_norm)]
    
    # compute reconstructions once — reuse for both PSNR and SSIM
    sup_recon = function_TV_denoising_CP(noisy_img, sup_min_alpha, 1000)
    sure_recon = function_TV_denoising_CP(noisy_img, sure_min_alpha, 1000)
    r2r_recon = function_TV_denoising_CP(noisy_img, r2r_min_alpha, 1000)
    n2n_recon = function_TV_denoising_CP(noisy_img, n2n_min_alpha, 1000)
    
    # compute PSNRs
    sup_psnr = psnr(sup_recon, clean_img)
    sure_psnr = psnr(sure_recon, clean_img)
    r2r_psnr = psnr(r2r_recon, clean_img)
    n2n_psnr = psnr(n2n_recon, clean_img)
    noisy_psnr = psnr(noisy_img, clean_img)
    
    # compute SSIMs
    sup_ssim = ssim(sup_recon, clean_img, data_range=1.0)
    sure_ssim = ssim(sure_recon, clean_img, data_range=1.0)
    r2r_ssim = ssim(r2r_recon, clean_img, data_range=1.0)
    n2n_ssim = ssim(n2n_recon, clean_img, data_range=1.0)
    noisy_ssim = ssim(noisy_img, clean_img, data_range=1.0)
    
    # plot and save
    plt.figure(figsize=(8, 5))
    plt.plot(plot_alphas, supervised_norm, color='orange', label='Supervised')
    plt.plot(plot_alphas, sure_norm, color='blue', label='SURE')
    plt.plot(plot_alphas, r2r_norm, color='green', label='R2R')
    plt.plot(plot_alphas, n2n_norm, color='red', label='N2N')
    plt.axvline(sup_min_alpha, color='orange', linestyle='--', alpha=0.7)
    plt.axvline(sure_min_alpha, color='blue', linestyle='--', alpha=0.7)
    plt.axvline(r2r_min_alpha, color='green', linestyle='--', alpha=0.7)
    plt.axvline(n2n_min_alpha, color='red', linestyle='--', alpha=0.7)
    plt.xlabel('alpha')
    plt.ylabel('Normalised loss')
    plt.title(f'Loss vs Alpha - {image_name}')
    plt.legend()
    plt.grid(True)
    plt.semilogx()
    plt.savefig(f'loss_plot_{image_name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # store results
    results = {
        'image': image_name,
        'alphas': {
            'supervised': sup_min_alpha,
            'SURE': sure_min_alpha,
            'R2R': r2r_min_alpha,
            'N2N': n2n_min_alpha
        },
        'psnrs': {
            'noisy': noisy_psnr,
            'supervised': sup_psnr,
            'SURE': sure_psnr,
            'R2R': r2r_psnr,
            'N2N': n2n_psnr
        },
        'ssims': {
            'noisy': noisy_ssim,
            'supervised': sup_ssim,
            'SURE': sure_ssim,
            'R2R': r2r_ssim,
            'N2N': n2n_ssim
        }
    }
    return results

all_results = []
for name in image_names:
    clean = load_image_256(f'archive/{name}')
    noisy = corrupt_image(clean, 0.05)
    results = run_experiment(noisy, clean, name)
    all_results.append(results)
    print(f"\n{name} results:")
    print(f"  Supervised: α={results['alphas']['supervised']:.4f}, PSNR={results['psnrs']['supervised']:.4f}, SSIM={results['ssims']['supervised']:.4f}")
    print(f"  SURE:       α={results['alphas']['SURE']:.4f}, PSNR={results['psnrs']['SURE']:.4f}, SSIM={results['ssims']['SURE']:.4f}")
    print(f"  R2R:        α={results['alphas']['R2R']:.4f}, PSNR={results['psnrs']['R2R']:.4f}, SSIM={results['ssims']['R2R']:.4f}")
    print(f"  N2N:        α={results['alphas']['N2N']:.4f}, PSNR={results['psnrs']['N2N']:.4f}, SSIM={results['ssims']['N2N']:.4f}")
    print(f"  Noisy:      PSNR={results['psnrs']['noisy']:.4f}, SSIM={results['ssims']['noisy']:.4f}")



Printing Out full Results

In [ ]:
# print all results
for r in all_results:
    print(f"\n{r['image']}:")
    print(f"  Alphas:  sup={r['alphas']['supervised']}, SURE={r['alphas']['SURE']}, R2R={r['alphas']['R2R']}, N2N={r['alphas']['N2N']}")
    print(f"  PSNRs:   sup={r['psnrs']['supervised']}, SURE={r['psnrs']['SURE']}, R2R={r['psnrs']['R2R']}, N2N={r['psnrs']['N2N']}, noisy={r['psnrs']['noisy']}")
    print(f"  SSIMs:   sup={r['ssims']['supervised']}, SURE={r['ssims']['SURE']}, R2R={r['ssims']['R2R']}, N2N={r['ssims']['N2N']}, noisy={r['ssims']['noisy']}")

Plotting Box Plot of Gaps between Supervised and Self-Supervised Alphas

In [ ]:
sure_gaps = [r['alphas']['supervised'] - r['alphas']['SURE'] for r in all_results]
r2r_gaps = [r['alphas']['supervised'] - r['alphas']['R2R'] for r in all_results]
n2n_gaps = [r['alphas']['supervised'] - r['alphas']['N2N'] for r in all_results]

plt.figure(figsize=(8,5))
plt.boxplot([sure_gaps, r2r_gaps, n2n_gaps], 
            labels=['SURE', 'R2R', 'N2N'],
            patch_artist=True,
            boxprops=dict(facecolor='lightblue'),
            medianprops=dict(color='red', linewidth=2))
plt.axhline(y=0, color='black', linestyle='--', alpha=0.7, label='Supervised baseline')
plt.ylabel('Supervised α - Method α')
plt.title('Distribution of Alpha Gaps Relative to Supervised Baseline')
plt.legend()
plt.grid(True)
plt.savefig('alpha_gaps_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

# also print summary statistics
print(f"SURE  - Mean gap: {np.mean(sure_gaps):.4f}, Std: {np.std(sure_gaps):.4f}")
print(f"R2R   - Mean gap: {np.mean(r2r_gaps):.4f}, Std: {np.std(r2r_gaps):.4f}")
print(f"N2N   - Mean gap: {np.mean(n2n_gaps):.4f}, Std: {np.std(n2n_gaps):.4f}")

Finding Mean for PSNR and SSIM

In [ ]:
import numpy as np

print("\nMean PSNRs across all images:")
print(f"  Supervised: {np.mean([r['psnrs']['supervised'] for r in all_results])}")
print(f"  SURE:       {np.mean([r['psnrs']['SURE'] for r in all_results])}")
print(f"  R2R:        {np.mean([r['psnrs']['R2R'] for r in all_results])}")
print(f"  N2N:        {np.mean([r['psnrs']['N2N'] for r in all_results])}")
print(f"  Noisy:      {np.mean([r['psnrs']['noisy'] for r in all_results])}")

print("\nMean SSIMs across all images:")
print(f"  Supervised: {np.mean([r['ssims']['supervised'] for r in all_results])}")
print(f"  SURE:       {np.mean([r['ssims']['SURE'] for r in all_results])}")
print(f"  R2R:        {np.mean([r['ssims']['R2R'] for r in all_results])}")
print(f"  N2N:        {np.mean([r['ssims']['N2N'] for r in all_results])}")
print(f"  Noisy:      {np.mean([r['ssims']['noisy'] for r in all_results])}")

Finding Standard Deviation for PSNR and SSIM

In [ ]:
print("\nStd PSNRs across all images:")
print(f"  Supervised: {np.std([r['psnrs']['supervised'] for r in all_results])}")
print(f"  SURE:       {np.std([r['psnrs']['SURE'] for r in all_results])}")
print(f"  R2R:        {np.std([r['psnrs']['R2R'] for r in all_results])}")
print(f"  N2N:        {np.std([r['psnrs']['N2N'] for r in all_results])}")
print(f"  Noisy:      {np.std([r['psnrs']['noisy'] for r in all_results])}")

print("\nStd SSIMs across all images:")
print(f"  Supervised: {np.std([r['ssims']['supervised'] for r in all_results])}")
print(f"  SURE:       {np.std([r['ssims']['SURE'] for r in all_results])}")
print(f"  R2R:        {np.std([r['ssims']['R2R'] for r in all_results])}")
print(f"  N2N:        {np.std([r['ssims']['N2N'] for r in all_results])}")
print(f"  Noisy:      {np.std([r['ssims']['noisy'] for r in all_results])}")

Creating PSNR Plots

In [ ]:
# ---- PSNR plots ----
images_to_plot = [
    {
        'name': 'kodim21.png',
        'supervised': 0.029510028653295134,
        'SURE': 0.027799426934097426,
        'R2R': 0.047756446991404015,
        'N2N': 0.04205444126074499
    },
    {
        'name': 'kodim05.png',
        'supervised': 0.0226676217765043,
        'SURE': 0.025518624641833815,
        'R2R': 0.03863323782234958,
        'N2N': 0.05459885386819485
    },
    {
        'name': 'kodim08.png',
        'supervised': 0.0220974212034384,
        'SURE': 0.023808022922636108,
        'R2R': 0.03863323782234958,
        'N2N': 0.05630945558739256
    }
]

for img_data in images_to_plot:
    clean_test_img = load_image_256(f'archive/{img_data["name"]}')
    noisy_img = corrupt_image(clean_test_img, 0.05)
    
    selected_alphas = {
        'Supervised': img_data['supervised'],
        'SURE': img_data['SURE'],
        'R2R': img_data['R2R'],
        'N2N': img_data['N2N']
    }
    
    plot_alphas = np.linspace(0.001, 0.2, 500)
    PSNRs_dense = [psnr(function_TV_denoising_CP(noisy_img, a, 1000), clean_test_img) 
                   for a in plot_alphas]
    
    selected_psnrs = {
        'Supervised': psnr(function_TV_denoising_CP(noisy_img, img_data['supervised'], 1000), clean_test_img),
        'SURE': psnr(function_TV_denoising_CP(noisy_img, img_data['SURE'], 1000), clean_test_img),
        'R2R': psnr(function_TV_denoising_CP(noisy_img, img_data['R2R'], 1000), clean_test_img),
        'N2N': psnr(function_TV_denoising_CP(noisy_img, img_data['N2N'], 1000), clean_test_img)
    }
    
    plot_psnr(plot_alphas, PSNRs_dense, selected_alphas, selected_psnrs)
    plt.savefig(f'psnr_plot_{img_data["name"]}', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

def add_zoom_inset(ax, image, zoom_region, inset_loc='lower right'):
    x, y, w, h = zoom_region
    axins = inset_axes(ax, width='40%', height='40%', loc=inset_loc)
    axins.imshow(image, cmap='gray')
    axins.set_xlim(x, x+w)
    axins.set_ylim(y+h, y)
    axins.set_xticks([])
    axins.set_yticks([])
    mark_inset(ax, axins, loc1=1, loc2=2, fc='none', ec='red', linewidth=1.5)

def plot_reconstructions(clean, noisy, sup_recon, sure_recon, r2r_recon, n2n_recon,
                         sup_alpha, sure_alpha, r2r_alpha, n2n_alpha,
                         zoom, save_name, inset_loc='lower right'):
    
    noisy_psnr_val = psnr(noisy, clean)
    sup_psnr_val = psnr(sup_recon, clean)
    sure_psnr_val = psnr(sure_recon, clean)
    r2r_psnr_val = psnr(r2r_recon, clean)
    n2n_psnr_val = psnr(n2n_recon, clean)

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    images = [clean, noisy, sup_recon, sure_recon, r2r_recon, n2n_recon]
    titles = [
        'Clean $u_{true}$',
        f'Noisy\nPSNR: {noisy_psnr_val:.2f} dB',
        f'Supervised\nPSNR: {sup_psnr_val:.2f} dB, $\\alpha$={sup_alpha:.4f}',
        f'SURE\nPSNR: {sure_psnr_val:.2f} dB, $\\alpha$={sure_alpha:.4f}',
        f'R2R\nPSNR: {r2r_psnr_val:.2f} dB, $\\alpha$={r2r_alpha:.4f}',
        f'N2N\nPSNR: {n2n_psnr_val:.2f} dB, $\\alpha$={n2n_alpha:.4f}'
    ]

    for ax, img, title in zip(axes.flat, images, titles):
        ax.imshow(img, cmap='gray')
        ax.set_title(title, fontsize=12)
        ax.axis('off')
        add_zoom_inset(ax, img, zoom, inset_loc)

    plt.tight_layout()
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()

Creating Reconstruction Plots for Select Images

In [ ]:
# ---- Reconstruction plots ----

# kodim21
clean = load_image_256('archive/kodim21.png')
noisy = corrupt_image(clean, 0.05)
sup_recon = function_TV_denoising_CP(noisy, 0.029510028653295134, 1000)
sure_recon = function_TV_denoising_CP(noisy, 0.027799426934097426, 1000)
r2r_recon = function_TV_denoising_CP(noisy, 0.047756446991404015, 1000)
n2n_recon = function_TV_denoising_CP(noisy, 0.04205444126074499, 1000)

plot_reconstructions(clean, noisy, sup_recon, sure_recon, r2r_recon, n2n_recon,
                     0.029510028653295134, 0.027799426934097426,
                     0.047756446991404015, 0.04205444126074499,
                     zoom=[10, 90, 90, 60],
                     save_name='reconstructions_kodim21.png')

# kodim05
clean = load_image_256('archive/kodim05.png')
noisy = corrupt_image(clean, 0.05)
sup_recon = function_TV_denoising_CP(noisy, 0.0226676217765043, 1000)
sure_recon = function_TV_denoising_CP(noisy, 0.025518624641833815, 1000)
r2r_recon = function_TV_denoising_CP(noisy, 0.03863323782234958, 1000)
n2n_recon = function_TV_denoising_CP(noisy, 0.05459885386819485, 1000)

plot_reconstructions(clean, noisy, sup_recon, sure_recon, r2r_recon, n2n_recon,
                     0.0226676217765043, 0.025518624641833815,
                     0.03863323782234958, 0.05459885386819485,
                     zoom=[50, 50, 80, 80],  # adjust for kodim05
                     save_name='reconstructions_kodim05.png')

# kodim08
clean = load_image_256('archive/kodim08.png')
noisy = corrupt_image(clean, 0.05)
sup_recon = function_TV_denoising_CP(noisy, 0.0220974212034384, 1000)
sure_recon = function_TV_denoising_CP(noisy, 0.023808022922636108, 1000)
r2r_recon = function_TV_denoising_CP(noisy, 0.03863323782234958, 1000)
n2n_recon = function_TV_denoising_CP(noisy, 0.05630945558739256, 1000)

plot_reconstructions(clean, noisy, sup_recon, sure_recon, r2r_recon, n2n_recon,
                     0.0220974212034384, 0.023808022922636108,
                     0.03863323782234958, 0.05630945558739256,
                     zoom=[0, 60, 80, 80],  # adjust for kodim08
                     save_name='reconstructions_kodim08.png')

Spatially Varying Experiments


In [ ]:
clean_test_img = load_image_256('archive/kodim21.png')
noisy_img = corrupt_image(clean_test_img, 0.05)

In [ ]:

image_blocks = break_image(noisy_img, 16)
denoised_blocks, spatially_varying_alphas = find_spatially_varying_alpha_supervised(image_blocks, clean_test_img)
reconstruction = rebuild_image(denoised_blocks, 16)
plt.figure(figsize = (7,7)) 
imgplot2 = plt.imshow(reconstruction)
imgplot2.set_cmap('gray')

In [ ]:
def plot_alpha_heatmap(spatially_varying_alpha, image_name):
    alpha_grid = np.array(spatially_varying_alpha).reshape(16, 16)
    
    plt.figure(figsize=(6,5))
    plt.imshow(alpha_grid, cmap='hot', interpolation='nearest')
    plt.colorbar(label='Alpha value')
    plt.title(f'Spatially Varying Alpha Heatmap - {image_name}')
    plt.savefig(f'alpha_heatmap_{image_name}', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Min alpha: {alpha_grid.min():.4f}")
    print(f"Max alpha: {alpha_grid.max():.4f}")
    print(f"Mean alpha: {alpha_grid.mean():.4f}")


plot_alpha_heatmap(spatially_varying_alphas, 'kodim21.png')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(noisy_img, cmap='gray')
axes[0].set_title('Noisy image')

axes[1].imshow(reconstruction, cmap='gray')
axes[1].set_title(f'Spatially varying reconstruction\nPSNR: {psnr(reconstruction, clean_test_img):.4f}')

axes[2].imshow(np.array(spatially_varying_alphas).reshape(16,16),
               cmap='hot', interpolation='nearest')
axes[2].set_title('Alpha heatmap')
plt.colorbar(axes[2].images[0], ax=axes[2], label='Alpha value')

plt.savefig('spatially_varying_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
supervised_reconstruction_alpha = find_optimal_alpha(noisy_img,clean_test_img)[0]
supervised_reconstruction = function_TV_denoising_CP(noisy_img, supervised_reconstruction_alpha, 1000)
print(psnr(supervised_reconstruction, clean_test_img))

Supervised Spatially Varying Experiments

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

axes[0,0].imshow(noisy_img, cmap='gray')
axes[0,0].set_title(f'Noisy image\nPSNR: {psnr(noisy_img, clean_test_img):.4f}')
axes[0,0].axis('off')

axes[0,1].imshow(supervised_reconstruction, cmap='gray')
axes[0,1].set_title(f'Global supervised\nPSNR: {psnr(supervised_reconstruction, clean_test_img):.4f}')
axes[0,1].axis('off')

axes[1,0].imshow(reconstruction, cmap='gray')
axes[1,0].set_title(f'Spatially varying\nPSNR: {psnr(reconstruction, clean_test_img):.4f}')
axes[1,0].axis('off')

im = axes[1,1].imshow(np.array(spatially_varying_alphas).reshape(16,16),
               cmap='hot', interpolation='nearest')
axes[1,1].set_title('Alpha heatmap')
axes[1,1].axis('off')
plt.colorbar(im, ax=axes[1,1], label='Alpha value')

plt.tight_layout()
plt.savefig('spatially_varying_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

Self Supervised Spatially Varying Tests

In [ ]:
image_blocks = break_image(noisy_img, 16)
denoised_blocks, spatially_varying_alphas = find_spatially_varying_alpha_SURE(image_blocks)
reconstruction = rebuild_image(denoised_blocks, 16)
plt.figure(figsize = (7,7)) 
imgplot2 = plt.imshow(reconstruction)
imgplot2.set_cmap('gray')

In [ ]:
def plot_alpha_heatmap(spatially_varying_alpha, image_name):
    alpha_grid = np.array(spatially_varying_alpha).reshape(16, 16)
    
    plt.figure(figsize=(6,5))
    plt.imshow(alpha_grid, cmap='hot', interpolation='nearest')
    plt.colorbar(label='Alpha value')
    plt.title(f'Spatially Varying Alpha Heatmap - {image_name}')
    plt.savefig(f'alpha_heatmap_{image_name}', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Min alpha: {alpha_grid.min():.4f}")
    print(f"Max alpha: {alpha_grid.max():.4f}")
    print(f"Mean alpha: {alpha_grid.mean():.4f}")


plot_alpha_heatmap(spatially_varying_alphas, 'kodim21.png')

In [ ]:
test_images = ['kodim01.png', 'kodim06.png', 'kodim15.png', 'kodim23.png', 'kodim21.png']

for name in test_images:
    clean = load_image_256(f'archive/{name}')
    noisy = corrupt_image(clean, 0.05)
    
    # compute global supervised alpha for this image
    global_alpha = find_optimal_alpha(noisy, clean)[0]
    global_recon = function_TV_denoising_CP(noisy, global_alpha, 1000)
    
    # compute spatially varying
    image_blocks = break_image(noisy, 16)
    denoised_blocks, alphas = find_spatially_varying_alpha_supervised(image_blocks, clean)
    reconstruction_sup = rebuild_image(denoised_blocks, 16)

    #compute spatially varying SURE
    denoised_blocks, alphas = find_spatially_varying_alpha_SURE(image_blocks)
    reconstruction_sure = rebuild_image(denoised_blocks, 16)
    
    print(f"\n{name}:")
    print(f"  Global supervised alpha: {global_alpha:.4f}")
    print(f"  Global supervised PSNR: {psnr(global_recon, clean):.4f}")
    print(f"  Spatially varying SURE PSNR: {psnr(reconstruction_sure, clean):.4f}")
    print(f"  PSNR improvement: {psnr(reconstruction_sure, clean) - psnr(global_recon, clean):.4f} dB")
    
    # plot all four side by side
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    axes[0].imshow(noisy, cmap='gray')
    axes[0].set_title(f'Noisy\nPSNR: {psnr(noisy, clean):.4f}')
    axes[0].axis('off')
    
    axes[1].imshow(global_recon, cmap='gray')
    axes[1].set_title(f'Global Supervised\nα={global_alpha:.4f}, PSNR: {psnr(global_recon, clean):.4f}')
    axes[1].axis('off')
    
    axes[2].imshow(reconstruction_sup, cmap='gray')
    axes[2].set_title(f'Spatially Varying Supervised\nPSNR: {psnr(reconstruction_sup, clean):.4f}')
    axes[2].axis('off')
    
    axes[3].imshow(reconstruction_sure, cmap='gray')
    axes[3].set_title(f'Spatially Varying SURE\nPSNR: {psnr(reconstruction_sure, clean):.4f}')
    axes[3].axis('off')
    
    
    plt.suptitle(f'{name}', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'spatially_varying_{name}', dpi=150, bbox_inches='tight')
    plt.show()